In [ ]:
from impact import Impact
from impact.model.distgen.distgen_impact_model import LUMEDistgenImpactModel
from impact.model.actions import ImpactVarAction, EleVarAction
from impact.model.config import (
    VariableMappingConfig,
    ElementsConfig,
    StatsConfig,
    ParticlesConfig,
    RunInfoConfig,
)
from distgen import Generator

from lume.variables import ScalarVariable, NDVariable, ParticleGroupVariable
import matplotlib.pyplot as plt

In [ ]:
# Load the distgen generator object
gen = Generator("templates/lcls_injector/distgen.yaml")
gen["n_particle"] = 10000

In [ ]:
# Load LCLS Impact-T lattice
imp = Impact("templates/lcls_injector/ImpactT.in")
imp.header["Np"] = 10000
imp.header["Nx"] = 16
imp.header["Ny"] = 16
imp.header["Nz"] = 16
imp.header["Dt"] = 5e-13

# Turn Space Charge off. Both these syntaxes work
imp.header["Bcurr"] = 0
imp["header:Bcurr"] = 0

# Other switches
imp.timeout = None

# Switches for MPI
imp.numprocs = 0  # Auto-select

imp.run()

In [ ]:
# Dict with element name mappings from SLAC machine name to impact name
ele_name_mappings = {"QUAD:IN20:425": "QE01"}

# Configure automatic variable and action creation from Impact-T instance
imp_config = VariableMappingConfig(
    elements=ElementsConfig(
        pattern="{name}:{attrib}",
        name_mappings=ele_name_mappings,
        regex=r"^(?P<name>.+):(?P<attrib>[^:]+)$",  # Regex to capture item after last colon as attribute, everything else as name
    ),
    stats=StatsConfig(pattern="STATS:{name}"),
    particles=ParticlesConfig(pattern="PAR:{name}"),
    run_info=RunInfoConfig(pattern="RUNINFO:{key}"),
)

In [ ]:
# Generate model automatically creating
model = LUMEDistgenImpactModel.from_objects(
    gen,
    imp,
    impact_config=imp_config,
)


# Show default loaded variables, can control names and inclusion of all attributes from `VariableMappingConfig` object
[var for var in model.supported_variables.values() if "IN20" in var.name]

In [ ]:
# Look at the stats keys
[
    (var.name, var.shape)
    for var in model.supported_variables.values()
    if isinstance(var, NDVariable) and var.read_only
]

In [ ]:
# Look at the particles keys
[
    var.name
    for var in model.supported_variables.values()
    if isinstance(var, ParticleGroupVariable)
]

In [ ]:
# Example of user extended mapping with scale and offset
class ScaledEleVariableMapping(ImpactVarAction):
    var: ScalarVariable
    tool_name: str
    tool_attrib: str

    scale: float
    offset: float

    def get(self, imp):
        return self.scale * imp.ele[self.tool_name][self.tool_attrib] + self.offset

    def set(self, imp, value):
        imp.ele[self.tool_name][self.tool_attrib] = (value - self.offset) / self.scale


# Override the existing quad mappings with a 10% scaling factor
for mapping in model.impact_actions:
    if isinstance(mapping, EleVarAction):
        model.register_action(
            ScaledEleVariableMapping(
                var=mapping.var,
                tool_name=mapping.tool_name,
                tool_attrib=mapping.tool_attrib,
                scale=1.1,
                offset=0.0,
            )
        )

In [ ]:
# Add custom attribute to the quads
class BCTRLVarAction(ImpactVarAction):
    tool_name: str

    def get(self, imp):
        integral = (
            imp.ele[self.tool_name]["b1_gradient"] * imp.ele[self.tool_name]["L"]
        )  # Example calc, use own model to convert
        print(f"Read ({self.tool_name}) integral = {integral:.2f}")
        return integral

    def set(self, imp, value):
        b1_grad = (
            value / imp.ele[self.tool_name]["L"]
        )  # Example calc, use own model to convert
        print(f"Setting ({self.tool_name}) b1_grad = {b1_grad:.2f}")
        imp.ele[self.tool_name]["b1_gradient"] = b1_grad


# Add custom variable for each quad (SLAC BCTRL)
inv_name_map = {v: k for k, v in ele_name_mappings.items()}
ctrl_vars = []
for ele in imp.lattice:
    name = ele["name"]
    if ele["type"] == "quadrupole":
        model.register_action(
            BCTRLVarAction(
                var=ScalarVariable(
                    name=inv_name_map.get(name, name) + ":BCTRL", unit="kG"
                ),
                tool_name=name,
            )
        )

In [ ]:
class DipoleStringVarAction(ImpactVarAction):
    dipoles_names: list[str]

    def set(self, imp, value):
        for name in self.dipoles_names:
            imp.ele[name]["strength"] = value

In [ ]:
# Get the stats object
stats = model.get(["STATS:sigma_x", "STATS:mean_z", "STATS:mean_kinetic_energy"])
plt.plot(stats["STATS:mean_z"], 1e6 * stats["STATS:sigma_x"], c="C0")
plt.xlabel("s (m)")
plt.ylabel("RMS Beam Size (um, Blue)")
plt.twinx()
plt.plot(stats["STATS:mean_z"], 1e-6 * stats["STATS:mean_kinetic_energy"], c="C1")
plt.ylabel("Beam Energy (MeV, Orange)")

In [ ]:
# Turn down the gun's field
model.set({"GUN:rf_field_scale": 32291431.56499891})

# Get the stats object
stats = model.get(["STATS:sigma_x", "STATS:mean_z", "STATS:mean_kinetic_energy"])
plt.plot(stats["STATS:mean_z"], 1e6 * stats["STATS:sigma_x"], c="C0")
plt.xlabel("s (m)")
plt.ylabel("RMS Beam Size (um, Blue)")
plt.twinx()
plt.plot(stats["STATS:mean_z"], 1e-6 * stats["STATS:mean_kinetic_energy"], c="C1")
plt.ylabel("Beam Energy (MeV, Orange)")